# VisA defect classification with ResNet-18

Train a binary classifier on candle images using the official VisA two-class high-shot split. The pretrained ResNet-18 stays frozen while its new final layer learns to predict defects. This supervised run uses all labels in the training split.

[Open in Colab](https://colab.research.google.com/github/Brafied/defect-detection-active-learning/blob/resnet18-baseline/visa_resnet18_baseline.ipynb)

### 1. Load the candle split

The first run downloads the [VisA archive](https://github.com/amazon-science/spot-diff) (about 1.9 GB) and extracts the selected product. Rerunning these cells reuses the local images. A Colab GPU is optional.

In [ ]:
import csv
from pathlib import Path
import subprocess

item = "candle"
data_dir = Path("visa_data")
data_dir.mkdir(exist_ok=True)
split_file = data_dir / "2cls_highshot.csv"
archive = data_dir / "VisA_20220922.tar"

def download(url, destination):
    if destination.is_file():
        return
    partial = destination.with_suffix(destination.suffix + ".part")
    subprocess.run(["curl", "--fail", "--location", "--retry", "2", "--output", str(partial), url], check=True)
    partial.replace(destination)

download("https://raw.githubusercontent.com/amazon-science/spot-diff/main/split_csv/2cls_highshot.csv", split_file)
with split_file.open(newline="") as file:
    rows = [row for row in csv.DictReader(file) if row["object"] == item]

if not rows:
    raise ValueError(f"No images found for product {item!r}")
if any(not (data_dir / row["image"]).is_file() for row in rows):
    download("https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar", archive)
    subprocess.run(["tar", "-xf", str(archive), "-C", str(data_dir), item], check=True)

### 2. Prepare the images and model

In [ ]:
import torch
from PIL import Image
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet18_Weights, resnet18

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = ResNet18_Weights.DEFAULT
preprocess = weights.transforms()

class VisaDatasetSplit(Dataset):
    def __init__(self, split):
        self.examples = [row for row in rows if row["split"] == split]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        row = self.examples[index]
        with Image.open(data_dir / row["image"]) as image:
            pixels = preprocess(image.convert("RGB"))
        label = torch.tensor({"normal": 0, "anomaly": 1}[row["label"]], dtype=torch.float32)
        return pixels, label

training_split = VisaDatasetSplit("train")
testing_split = VisaDatasetSplit("test")
training_loader = DataLoader(training_split, batch_size=16, shuffle=True)
testing_loader = DataLoader(testing_split, batch_size=16)
print(f"{item}: {len(training_split)} training and {len(testing_split)} test images")

In [ ]:
model = resnet18(weights=weights)
model.requires_grad_(False)
model.fc = nn.Linear(model.fc.in_features, 1)
model.to(device)
# Keep the frozen backbone's batch normalization statistics fixed during training.
model.eval()

loss_function = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.fc.parameters(), lr=1e-5, weight_decay=1e-4)

### 3. Train the final layer and evaluate

The head outputs one score per image. A defect probability of 0.5 or higher counts as a defective prediction.

In [ ]:
epoch_count = 10

for epoch in range(epoch_count):
    training_loss = 0.0
    for images, labels in training_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = loss_function(logits, labels)
        loss.backward()
        optimizer.step()
        training_loss += loss.item() * len(labels)

    correct = 0
    with torch.no_grad():
        for images, labels in testing_loader:
            logits = model(images.to(device)).squeeze(1)
            predictions = torch.sigmoid(logits).cpu() >= 0.5
            correct += (predictions == labels.bool()).sum().item()

    print(f"Epoch {epoch + 1}/{epoch_count} | "
          f"train loss {training_loss / len(training_split):.4f} | "
          f"test accuracy {correct / len(testing_split):.4f}")